<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L12_Zero_shot_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L12: Zero-shot Learning - Classification Without Training Examples

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 12 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand zero-shot learning concepts and capabilities
- Master zero-shot classification techniques
- Learn task formulation for unseen tasks
- Design effective prompts for zero-shot scenarios
- Use pre-trained models for zero-shot text classification
- Apply Natural Language Inference (NLI) for classification
- Evaluate zero-shot performance on custom datasets

---

## Concept: What is Zero-shot Learning?

**Zero-shot learning** is the ability of a model to perform tasks it has never been explicitly trained on, using only natural language instructions.

### Why Zero-shot Learning Matters:

**The Traditional Approach:**
- Collect labeled training data
- Train or fine-tune a model
- Deploy for specific task
- Repeat for each new task

**The Zero-shot Approach:**
- Use pre-trained model as-is
- Describe task in natural language
- Get predictions immediately
- No training data required

### Key Advantages:

1. **No Training Data Needed** - Work with tasks you have no examples for
2. **Instant Deployment** - No training time required
3. **Flexible** - Easily adapt to new categories or tasks
4. **Cost-Effective** - No annotation or training costs
5. **Rapid Prototyping** - Test ideas quickly

### How It Works:

Pre-trained language models learn general patterns during training on massive text corpora. This knowledge transfers to new tasks through:

- **Task Description** - Explaining what you want in natural language
- **Label Semantics** - Using meaningful category names
- **Prompt Design** - Structuring inputs effectively

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers torch accelerate datasets -q

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Zero-shot Classification with Pipelines

**Zero-shot classification** allows you to classify text into categories without training examples.

### The Zero-shot Classification Pipeline:

HuggingFace provides a convenient pipeline that uses Natural Language Inference (NLI) models for zero-shot classification.

**How it works:**
1. Takes your text and candidate labels
2. Formulates as NLI hypothesis: "This text is about [label]"
3. Computes entailment scores for each label
4. Returns labels ranked by confidence

### Key Parameters:

- **candidate_labels** - List of possible categories
- **multi_label** - Whether text can belong to multiple categories
- **hypothesis_template** - Custom template for formulating hypotheses

---

In [ ]:
# Step 2: Load Zero-shot Classification Pipeline

print("Loading Zero-shot Classification Pipeline\n")
print("=" * 60)

# Load the zero-shot classification pipeline
# Uses facebook/bart-large-mnli by default (trained on NLI)
classifier = pipeline("zero-shot-classification", device=0 if torch.cuda.is_available() else -1)

print("Pipeline loaded successfully!")
print(f"Model: {classifier.model.name_or_path}")
print(f"Task: {classifier.task}")
print()

print("This pipeline can classify text into ANY categories you define!")
print("No training required - just provide the labels you want.")

In [ ]:
# Step 3: Basic Zero-shot Classification

print("Basic Zero-shot Classification\n")
print("=" * 60)

# Example 1: Sentiment analysis
text1 = "I absolutely loved this movie! The acting was superb and the plot kept me engaged throughout."
labels1 = ["positive", "negative", "neutral"]

print("\nExample 1: Sentiment Analysis")
print("-" * 60)
print(f"Text: {text1}")
print(f"Candidate labels: {labels1}\n")

result1 = classifier(text1, candidate_labels=labels1)

print(f"Prediction: {result1['labels'][0]}")
print(f"Confidence: {result1['scores'][0]:.4f}\n")
print("All scores:")
for label, score in zip(result1['labels'], result1['scores']):
    print(f"  {label}: {score:.4f}")

# Example 2: Topic classification
print("\n\nExample 2: Topic Classification")
print("-" * 60)

text2 = "The Federal Reserve announced a 0.25% interest rate increase to combat inflation."
labels2 = ["politics", "sports", "technology", "business", "entertainment"]

print(f"Text: {text2}")
print(f"Candidate labels: {labels2}\n")

result2 = classifier(text2, candidate_labels=labels2)

print(f"Prediction: {result2['labels'][0]}")
print(f"Confidence: {result2['scores'][0]:.4f}\n")
print("All scores:")
for label, score in zip(result2['labels'], result2['scores']):
    print(f"  {label}: {score:.4f}")

print("\n" + "=" * 60)
print("\nZero-shot classification works without any training data!")

## Part 2: Task Formulation for Zero-shot Learning

**Task formulation** is how you frame your problem for zero-shot classification.

### Key Principles:

1. **Clear Label Names**
   - Use descriptive, unambiguous labels
   - Avoid overlapping categories
   - Consider label semantics

2. **Appropriate Granularity**
   - Not too broad: "content" vs "not content"
   - Not too narrow: 100 specific subcategories
   - Find the right level of detail

3. **Hypothesis Templates**
   - Default: "This example is {}"
   - Custom: "This text is about {}"
   - Task-specific: "The sentiment is {}"

4. **Multi-label vs Single-label**
   - Single-label: One category per text
   - Multi-label: Multiple categories possible

---

In [ ]:
# Step 4: Task Formulation Examples

print("Task Formulation Strategies\n")
print("=" * 60)

text = "Python is a high-level programming language known for its simplicity and readability."

# Strategy 1: Broad categories
print("\n1. BROAD CATEGORIES")
print("-" * 60)
broad_labels = ["technical", "non-technical"]
result = classifier(text, candidate_labels=broad_labels)
print(f"Labels: {broad_labels}")
print(f"Prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Strategy 2: Specific categories
print("\n2. SPECIFIC CATEGORIES")
print("-" * 60)
specific_labels = ["programming", "mathematics", "physics", "biology", "history"]
result = classifier(text, candidate_labels=specific_labels)
print(f"Labels: {specific_labels}")
print(f"Prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Strategy 3: Very specific categories
print("\n3. VERY SPECIFIC CATEGORIES")
print("-" * 60)
detailed_labels = ["Python programming", "Java programming", "web development", "data science", "machine learning"]
result = classifier(text, candidate_labels=detailed_labels)
print(f"Labels: {detailed_labels}")
print(f"Prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Strategy 4: Custom hypothesis template
print("\n4. CUSTOM HYPOTHESIS TEMPLATE")
print("-" * 60)
labels = ["beginner-friendly", "advanced", "expert-level"]
result = classifier(
    text, 
    candidate_labels=labels,
    hypothesis_template="This programming language is {}"
)
print(f"Labels: {labels}")
print(f"Template: 'This programming language is {{}}'")
print(f"Prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

print("\n" + "=" * 60)
print("\nChoose label granularity and templates based on your task!")

## Part 3: Multi-label Zero-shot Classification

**Multi-label classification** allows text to belong to multiple categories simultaneously.

### When to Use Multi-label:

- Product categorization (electronics AND portable)
- Content tagging (tutorial AND beginner-friendly)
- Emotion detection (happy AND excited)
- Document classification (technical AND educational)

### How It Works:

Instead of picking the highest score, the model evaluates each label independently:
- Each label gets a probability
- Labels above threshold are selected
- Multiple labels can be true

---

In [ ]:
# Step 5: Multi-label Classification

print("Multi-label Zero-shot Classification\n")
print("=" * 60)

# Example: Product description
product_text = "Wireless Bluetooth headphones with noise cancellation, 30-hour battery life, and premium sound quality. Perfect for travel and commuting."

labels = ["electronics", "audio", "portable", "travel", "premium", "affordable"]

# Single-label (default)
print("\nSINGLE-LABEL MODE (multi_label=False)")
print("-" * 60)
print(f"Text: {product_text}\n")

result_single = classifier(product_text, candidate_labels=labels, multi_label=False)

print("Top prediction:")
print(f"  {result_single['labels'][0]}: {result_single['scores'][0]:.4f}\n")

print("All scores (sum to 1.0):")
for label, score in zip(result_single['labels'], result_single['scores']):
    print(f"  {label}: {score:.4f}")

# Multi-label
print("\n\nMULTI-LABEL MODE (multi_label=True)")
print("-" * 60)

result_multi = classifier(product_text, candidate_labels=labels, multi_label=True)

print("All scores (independent probabilities):")
for label, score in zip(result_multi['labels'], result_multi['scores']):
    print(f"  {label}: {score:.4f}")

# Select labels above threshold
threshold = 0.5
selected_labels = [label for label, score in zip(result_multi['labels'], result_multi['scores']) if score > threshold]

print(f"\nSelected labels (threshold={threshold}): {selected_labels}")

print("\n" + "=" * 60)
print("\nMulti-label allows multiple categories per text!")

## Part 4: Prompt Design for Zero-shot Tasks

**Prompt design** significantly impacts zero-shot performance.

### Effective Label Design:

1. **Use Natural Language**
   - Good: "positive sentiment", "negative sentiment"
   - Bad: "pos", "neg"

2. **Be Specific**
   - Good: "customer complaint", "product inquiry"
   - Bad: "type1", "type2"

3. **Avoid Ambiguity**
   - Good: "urgent", "not urgent"
   - Bad: "important" (ambiguous meaning)

4. **Consider Synonyms**
   - Test variations: "happy" vs "joyful" vs "positive"
   - Choose labels that match your domain

### Hypothesis Template Design:

The template determines how labels are framed:

```python
# Default
"This example is {}"

# Task-specific
"This text expresses {} sentiment"
"The topic of this article is {}"
"This email is a {} request"
```

---

In [ ]:
# Step 6: Prompt Design Comparison

print("Prompt Design Impact on Performance\n")
print("=" * 60)

text = "I need to cancel my subscription immediately. This is urgent!"

# Test 1: Generic labels
print("\n1. GENERIC LABELS")
print("-" * 60)
generic_labels = ["type A", "type B", "type C"]
result = classifier(text, candidate_labels=generic_labels)
print(f"Labels: {generic_labels}")
print(f"Top prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Test 2: Descriptive labels
print("\n2. DESCRIPTIVE LABELS")
print("-" * 60)
descriptive_labels = ["urgent request", "general inquiry", "feedback"]
result = classifier(text, candidate_labels=descriptive_labels)
print(f"Labels: {descriptive_labels}")
print(f"Top prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Test 3: Very specific labels
print("\n3. VERY SPECIFIC LABELS")
print("-" * 60)
specific_labels = ["urgent cancellation request", "billing question", "technical support", "general feedback"]
result = classifier(text, candidate_labels=specific_labels)
print(f"Labels: {specific_labels}")
print(f"Top prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Test 4: Custom hypothesis template
print("\n4. CUSTOM HYPOTHESIS TEMPLATE")
print("-" * 60)
labels = ["urgent", "not urgent"]
result = classifier(
    text,
    candidate_labels=labels,
    hypothesis_template="This customer message is {}"
)
print(f"Labels: {labels}")
print(f"Template: 'This customer message is {{}}'")
print(f"Top prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")

print("\n" + "=" * 60)
print("\nDescriptive labels and custom templates improve accuracy!")

## Part 5: Real-world Zero-shot Applications

Zero-shot learning excels in scenarios where:
- You have no labeled data
- Categories change frequently
- You need rapid prototyping
- Annotation is expensive

### Common Use Cases:

1. **Content Moderation** - Detect policy violations
2. **Customer Support** - Route tickets to departments
3. **Product Categorization** - Classify new products
4. **Intent Detection** - Understand user requests
5. **Sentiment Analysis** - Gauge customer satisfaction
6. **News Classification** - Categorize articles
7. **Email Filtering** - Organize inbox automatically

---

In [ ]:
# Step 7: Real-world Application Examples

print("Real-world Zero-shot Applications\n")
print("=" * 60)

# Application 1: Content Moderation
print("\n1. CONTENT MODERATION")
print("-" * 60)

comments = [
    "Great product! Highly recommend it.",
    "This is spam! Click here for free money!!!",
    "I hate this company and everyone who works there.",
    "Can someone help me with installation?"
]

moderation_labels = ["appropriate", "spam", "harassment", "help request"]

for i, comment in enumerate(comments, 1):
    result = classifier(comment, candidate_labels=moderation_labels)
    print(f"\nComment {i}: {comment}")
    print(f"Category: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Application 2: Customer Support Routing
print("\n\n2. CUSTOMER SUPPORT TICKET ROUTING")
print("-" * 60)

tickets = [
    "My payment was declined but I was still charged",
    "How do I reset my password?",
    "The app crashes every time I try to upload a photo",
    "I want to upgrade to the premium plan"
]

departments = ["billing", "account management", "technical support", "sales"]

for i, ticket in enumerate(tickets, 1):
    result = classifier(ticket, candidate_labels=departments)
    print(f"\nTicket {i}: {ticket}")
    print(f"Route to: {result['labels'][0]} ({result['scores'][0]:.4f})")

# Application 3: Product Categorization
print("\n\n3. E-COMMERCE PRODUCT CATEGORIZATION")
print("-" * 60)

products = [
    "Organic cotton t-shirt, available in multiple colors",
    "Wireless gaming mouse with RGB lighting",
    "Stainless steel water bottle, 32oz capacity",
    "Mystery thriller novel by bestselling author"
]

categories = ["clothing", "electronics", "home goods", "books", "toys"]

for i, product in enumerate(products, 1):
    result = classifier(product, candidate_labels=categories)
    print(f"\nProduct {i}: {product}")
    print(f"Category: {result['labels'][0]} ({result['scores'][0]:.4f})")

print("\n" + "=" * 60)
print("\nZero-shot classification works across diverse domains!")

## Part 6: Zero-shot Learning on Custom Datasets

**Evaluating zero-shot performance** on your own data helps assess model suitability.

### Evaluation Process:

1. **Prepare Dataset**
   - Collect representative samples
   - Define clear categories
   - Create test set

2. **Run Predictions**
   - Apply zero-shot classifier
   - Collect predictions and scores

3. **Analyze Results**
   - Calculate accuracy
   - Identify confusion patterns
   - Iterate on labels/templates

4. **Compare Approaches**
   - Test different label formulations
   - Try various hypothesis templates
   - Evaluate multi-label vs single-label

---

In [ ]:
# Step 8: Custom Dataset Evaluation

print("Zero-shot Learning on Custom Dataset\n")
print("=" * 60)

# Create a custom dataset
custom_dataset = [
    {"text": "The stock market reached new highs today", "true_label": "business"},
    {"text": "Scientists discover new species in the Amazon", "true_label": "science"},
    {"text": "Local team wins championship game", "true_label": "sports"},
    {"text": "New smartphone features AI-powered camera", "true_label": "technology"},
    {"text": "President announces new economic policy", "true_label": "politics"},
    {"text": "Box office hit breaks records this weekend", "true_label": "entertainment"},
    {"text": "Researchers develop breakthrough cancer treatment", "true_label": "science"},
    {"text": "Tech giant reports quarterly earnings", "true_label": "business"},
]

labels = ["business", "science", "sports", "technology", "politics", "entertainment"]

print(f"Dataset size: {len(custom_dataset)} samples")
print(f"Categories: {labels}\n")

# Run predictions
predictions = []
true_labels = []

print("Running predictions...")
print("-" * 60)

for item in custom_dataset:
    result = classifier(item["text"], candidate_labels=labels)
    predicted_label = result["labels"][0]
    confidence = result["scores"][0]
    
    predictions.append(predicted_label)
    true_labels.append(item["true_label"])
    
    match = "✓" if predicted_label == item["true_label"] else "✗"
    print(f"{match} True: {item['true_label']:15} | Pred: {predicted_label:15} | Conf: {confidence:.4f}")
    print(f"  Text: {item['text'][:60]}...")

# Calculate accuracy
correct = sum(1 for pred, true in zip(predictions, true_labels) if pred == true)
accuracy = correct / len(predictions)

print("\n" + "=" * 60)
print(f"\nAccuracy: {accuracy:.2%} ({correct}/{len(predictions)} correct)")

# Confusion analysis
print("\nPer-category Performance:")
print("-" * 60)

for label in labels:
    label_true = [t for t in true_labels if t == label]
    label_correct = sum(1 for pred, true in zip(predictions, true_labels) if true == label and pred == true)
    
    if len(label_true) > 0:
        label_acc = label_correct / len(label_true)
        print(f"{label:15}: {label_acc:.2%} ({label_correct}/{len(label_true)})")

print("\n" + "=" * 60)
print("\nZero-shot achieves reasonable accuracy without any training!")

## Part 7: Limitations and Best Practices

### Limitations of Zero-shot Learning:

1. **Performance Gap**
   - Usually lower accuracy than fine-tuned models
   - Struggles with domain-specific jargon
   - May confuse similar categories

2. **Label Dependency**
   - Performance varies with label phrasing
   - Ambiguous labels cause confusion
   - Requires experimentation

3. **Context Understanding**
   - Limited by model's pre-training
   - May miss subtle nuances
   - Can be fooled by adversarial examples

4. **Computational Cost**
   - Slower than simple classifiers
   - Scales with number of labels
   - Requires significant resources

### Best Practices:

1. **Start Simple**
   - Begin with few, clear categories
   - Add complexity gradually
   - Test thoroughly

2. **Iterate on Labels**
   - Try different phrasings
   - Use domain-appropriate terms
   - Test with real examples

3. **Use Appropriate Models**
   - Larger models generally better
   - NLI-trained models for classification
   - Consider model size vs speed trade-offs

4. **Validate Performance**
   - Test on representative data
   - Monitor edge cases
   - Compare with baselines

5. **Know When to Fine-tune**
   - If accuracy is critical
   - When you have labeled data
   - For production systems

---

In [ ]:
# Step 9: Limitations Demonstration

print("Zero-shot Learning Limitations\n")
print("=" * 60)

# Limitation 1: Similar categories
print("\n1. CONFUSION WITH SIMILAR CATEGORIES")
print("-" * 60)

text = "I'm feeling okay, nothing special"
labels = ["happy", "joyful", "content", "satisfied", "pleased"]

result = classifier(text, candidate_labels=labels)
print(f"Text: {text}")
print(f"Labels: {labels}\n")
print("Predictions (very similar scores):")
for label, score in zip(result['labels'][:3], result['scores'][:3]):
    print(f"  {label}: {score:.4f}")

# Limitation 2: Domain-specific terms
print("\n\n2. DOMAIN-SPECIFIC JARGON")
print("-" * 60)

technical_text = "The patient presents with acute myocardial infarction"
general_labels = ["heart problem", "lung problem", "brain problem", "stomach problem"]

result = classifier(technical_text, candidate_labels=general_labels)
print(f"Text: {technical_text}")
print(f"Labels: {general_labels}")
print(f"Prediction: {result['labels'][0]} ({result['scores'][0]:.4f})")
print("Note: Model understands medical terms due to pre-training")

# Limitation 3: Label phrasing sensitivity
print("\n\n3. LABEL PHRASING SENSITIVITY")
print("-" * 60)

text = "This product is terrible and broke after one day"

# Test different label phrasings
label_sets = [
    ["good", "bad"],
    ["positive", "negative"],
    ["satisfied", "unsatisfied"],
    ["thumbs up", "thumbs down"]
]

print(f"Text: {text}\n")
for labels in label_sets:
    result = classifier(text, candidate_labels=labels)
    print(f"Labels: {labels:30} -> {result['labels'][0]} ({result['scores'][0]:.4f})")

print("\n" + "=" * 60)
print("\nBest Practices:")
print("  1. Use clear, unambiguous labels")
print("  2. Test with domain-specific examples")
print("  3. Iterate on label phrasing")
print("  4. Consider fine-tuning for critical applications")
print("  5. Validate on representative test data")

## Exercises

### Exercise 1: Custom Classification Task
Create a zero-shot classifier for a task of your choice:
```python
# Choose a domain (e.g., movie reviews, product feedback, news articles)
# Define 4-6 meaningful categories
# Collect 10-15 test examples
# Run zero-shot classification
# Calculate and analyze accuracy
```

### Exercise 2: Label Optimization
Optimize labels for better performance:
```python
text = "I can't believe how amazing this experience was!"
# Test at least 5 different label formulations:
# - ["positive", "negative"]
# - ["happy", "sad"]
# - ["satisfied", "unsatisfied"]
# - etc.
# Compare confidence scores and choose the best
```

### Exercise 3: Multi-label Classification
Build a multi-label classifier for product tags:
```python
products = [
    "Lightweight laptop with long battery life",
    "Organic cotton baby clothes, hypoallergenic",
    "Professional DSLR camera with 4K video"
]
# Define relevant tags (portable, eco-friendly, professional, etc.)
# Use multi_label=True
# Analyze which products get multiple tags
```

### Exercise 4: Hypothesis Template Experimentation
Test different hypothesis templates:
```python
text = "The customer service was unhelpful and rude"
labels = ["positive", "negative"]

# Test templates:
# - "This example is {}"
# - "The sentiment is {}"
# - "This review is {}"
# - "The customer feels {}"
# Compare results and confidence scores
```

### Exercise 5: Real-world Application
Build a complete zero-shot application:
```python
# Choose a use case (email routing, content moderation, etc.)
# Create a dataset of 20+ examples
# Define appropriate categories
# Implement zero-shot classification
# Calculate metrics (accuracy, per-class performance)
# Identify failure cases and iterate
```

### Exercise 6: Comparison with Prompting
Compare zero-shot classification with prompt-based approaches:
```python
# Use the same task with:
# 1. Zero-shot classification pipeline
# 2. Direct prompting with GPT-2 (from L11)
# Compare accuracy, speed, and ease of use
# Document trade-offs
```

---

## Key Takeaways

1. **Zero-shot learning** enables classification without training data
2. **NLI models** power zero-shot classification through entailment
3. **Label design** significantly impacts performance
4. **Hypothesis templates** customize how tasks are formulated
5. **Multi-label mode** allows multiple categories per text
6. **Descriptive labels** work better than generic codes
7. **Custom datasets** help evaluate zero-shot performance
8. **Limitations exist** - lower accuracy than fine-tuned models
9. **Best for** rapid prototyping and tasks without training data
10. **Iterate** on labels and templates for better results

### Quick Reference:

**Basic Zero-shot Classification:**
```python
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
result = classifier(
    "Your text here",
    candidate_labels=["label1", "label2", "label3"]
)
print(result["labels"][0])  # Top prediction
```

**Multi-label Classification:**
```python
result = classifier(
    "Your text here",
    candidate_labels=["label1", "label2", "label3"],
    multi_label=True
)
# Each label gets independent probability
```

**Custom Hypothesis Template:**
```python
result = classifier(
    "Your text here",
    candidate_labels=["label1", "label2"],
    hypothesis_template="This text is about {}"
)
```

**Batch Processing:**
```python
texts = ["text1", "text2", "text3"]
results = [
    classifier(text, candidate_labels=labels)
    for text in texts
]
```

### When to Use Zero-shot:

- **Use zero-shot when:**
  - No labeled training data available
  - Categories change frequently
  - Rapid prototyping needed
  - Annotation is expensive
  - Exploring new tasks

- **Consider fine-tuning when:**
  - High accuracy required
  - Labeled data available
  - Production deployment
  - Domain-specific terminology
  - Performance is critical

---

## Additional Resources

### Papers
- **"Zero-Shot Learning - A Comprehensive Evaluation"** (Xian et al., 2018) - Survey of zero-shot methods
- **"Entailment as Few-Shot Learner"** (Yin et al., 2019) - NLI for classification
- **"Language Models as Zero-Shot Learners"** (Radford et al., 2019) - GPT-2 zero-shot capabilities

### Documentation
- [HuggingFace Zero-shot Classification](https://huggingface.co/tasks/zero-shot-classification)
- [Zero-shot Pipeline Documentation](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline)
- [NLI Models on HuggingFace](https://huggingface.co/models?pipeline_tag=zero-shot-classification)

### Blog Posts
- [Zero-shot Learning in NLP](https://joeddav.github.io/blog/2020/05/29/ZSL.html)
- [Practical Zero-shot Classification](https://huggingface.co/blog/zero-shot-classification)
- [Zero-shot vs Few-shot Learning](https://www.ruder.io/state-of-transfer-learning-in-nlp/)

### Interactive Tools
- [Zero-shot Classification Demo](https://huggingface.co/spaces/zero-shot-classification)
- [NLI Model Playground](https://huggingface.co/facebook/bart-large-mnli)
- [Label Studio](https://labelstud.io/) - For creating test datasets

### Models to Try
- **facebook/bart-large-mnli** - Default, good balance
- **roberta-large-mnli** - Higher accuracy
- **deberta-v3-large-mnli** - State-of-the-art
- **MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli** - Multi-domain

---

## Next Lesson

**L13: Few-shot Learning** - Learn in-context learning, example selection strategies, and how to optimize few-shot prompting for better performance with minimal examples.

---

**Congratulations! You now understand zero-shot learning!**

*You've learned how to classify text into any categories without training data, design effective labels, and apply zero-shot learning to real-world tasks.*